In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Diabetes Prediction with the Pima Indians Dataset

## 1. Introduction
This project predicts the onset of diabetes from diagnostic measurements (glucose, BMI, age, blood pressure and others) using the Pima Indians Diabetes Database. About 35% of patients in the data are diabetic, so the classes are mildly imbalanced. The aim is to build a scikit-learn pipeline and test whether a complex model beats a simple one.

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/mragpavank/diabetes/diabetes.csv")
df.head()

In [ ]:
df.describe()

## 2. Data cleaning
Glucose, blood pressure, skin thickness, insulin and BMI cannot be 0 in a living person, so zeros in these columns were treated as missing values. Insulin is missing in about half the rows (374 of 768) and skin thickness in about 30% (227). Missing values are filled inside the pipeline, using the training data only, to avoid data leakage.

In [ ]:
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[cols] = df[cols].replace(0, np.nan)
df.isna().sum()

## 3. Modelling
The data was split 80/20 into training and test sets, stratified by outcome. Each model is a pipeline of imputation, random over-sampling of the minority class, and the classifier. Over-sampling happens only on the training folds during cross-validation. Two models were tuned with GridSearchCV (5-fold): gradient boosting and a logistic regression baseline.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns="Outcome")
y = df["Outcome"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_test.shape)

In [ ]:
from imblearn.pipeline import make_pipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

clf = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomOverSampler(random_state=42),
    GradientBoostingClassifier(random_state=42),
)

params = {
    "simpleimputer__strategy": ["mean", "median"],
    "gradientboostingclassifier__n_estimators": [50, 100, 200],
    "gradientboostingclassifier__max_depth": [2, 3, 4],
}

model = GridSearchCV(clf, param_grid=params, cv=5, n_jobs=-1, verbose=1)
model.fit(X_train, y_train)
print(model.best_params_)
print(model.best_score_)

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)


In [ ]:
best = model.best_estimator_.named_steps["gradientboostingclassifier"]
pd.Series(best.feature_importances_, index=X.columns).sort_values().plot(kind="barh")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

lr_clf = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomOverSampler(random_state=42),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)

lr_params = {
    "simpleimputer__strategy": ["mean", "median"],
    "logisticregression__C": [0.01, 0.1, 1, 10],
}

lr_model = GridSearchCV(lr_clf, param_grid=lr_params, cv=5, n_jobs=-1, verbose=1)
lr_model.fit(X_train, y_train)
print(lr_model.best_params_)
print(lr_model.best_score_)

In [ ]:
y_pred_lr = lr_model.predict(X_test)
print(classification_report(y_test, y_pred_lr))

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

results = pd.DataFrame({
    "Logistic Regression": [
        accuracy_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_lr),
    ],
    "Gradient Boosting": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
    ],
}, index=["Accuracy", "Precision (class 1)", "Recall (class 1)", "F1 (class 1)"]).round(3)

results

In [ ]:
print("Logistic Regression CV:", lr_model.best_score_)
print("Gradient Boosting CV:  ", model.best_score_)

## 4. Results
On the held-out test set (154 rows), gradient boosting scored higher than logistic regression (accuracy 0.766 vs 0.734, recall for diabetic patients 0.778 vs 0.704). However, cross-validated accuracy on the training data was identical (0.761 for both), and repeated cross-validation gave mean recall of 0.713 ± 0.07 for logistic regression and 0.702 ± 0.055 for gradient boosting.

In [ ]:
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
for name, m in [("Logistic Regression", lr_model.best_estimator_),
                ("Gradient Boosting", model.best_estimator_)]:
    scores = cross_val_score(m, X_train, y_train, cv=cv, scoring="recall")
    print(name, scores.mean().round(3), "+/-", scores.std().round(3))

## 5. Conclusion and limitations
Logistic regression and gradient boosting performed equivalently. Gradient boosting's higher test-set scores are best explained by the small test size, since a single patient changes recall by about 2 points. Because the simpler model performs as well, logistic regression is preferable: it is easier to interpret clinically, as each coefficient shows how a factor such as glucose or BMI shifts the odds of diabetes.

Limitations: small sample (768 rows), heavily missing insulin values, and a single train/test split. This is a learning project on a public dataset, not a clinical tool.